# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RehanTajik-730/ML_INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row now equals exactly one pseudonymized content item, per client, per day.

In [6]:
import duckdb
import pandas as pd

con = duckdb.connect()
parquet_path = 'data_0.parquet'

query_window = f"""
SELECT
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(*) as total_daily_rows
FROM read_parquet('{parquet_path}')
"""
print("--- Time Window Verification ---")
print(con.execute(query_window).df())

query_grain = f"""
WITH grain_check AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) as duplicate_count
    FROM read_parquet('{parquet_path}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
)
SELECT COUNT(*) as violation_count FROM grain_check
"""
print("\n--- Grain Verification ---")
violation_count = con.execute(query_grain).fetchone()[0]
if violation_count == 0:
    print("Grain verified: 0 duplicate rows found for (report_date, client_hash_id, content_hash_id).")
else:
    print(f"Warning: Found {violation_count} duplicate rows violating the grain.")

--- Time Window Verification ---
  start_date   end_date  total_daily_rows
0 2026-03-01 2026-03-31           9841378

--- Grain Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain verified: 0 duplicate rows found for (report_date, client_hash_id, content_hash_id).


## 2. Fields: feature / label / context / excluded

FEATURES: impressions, clicks, avg_position, content_age_days, sessions, engagement_rate, scroll_rate

LABEL: is_ctr_opportunity. Sinc it looks for current optimization candidates it is derived as a proxy.

CONTEXT: client_hash_id, content_hash_id, report_date

EXCLUDED: health_score, priority_score and action_type. These fields are FlyRank's existing, rule based productions. If fed in the model they will cause data leakage

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd

con = duckdb.connect()
parquet_path = 'data_0.parquet'

print("--- 1. Date Span & Row Count ---")
query_window = f"""
SELECT
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(*) as total_rows
FROM read_parquet('{parquet_path}')
"""
print(con.execute(query_window).df())

print("\n--- 2. The Grain Verification ---")
query_grain = f"""
WITH grain_check AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) as duplicate_count
    FROM read_parquet('{parquet_path}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
)
SELECT COUNT(*) as violation_count FROM grain_check
"""
violation_count = con.execute(query_grain).fetchone()[0]
if violation_count == 0:
    print("Grain verified: 0 duplicates found. One row exactly equals one content item per client per day.")
else:
    print(f"Warning: Found {violation_count} duplicates.")

print("\n--- 3. Availability (Survival Check) ---")
query_availability = f"""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4,
    ROUND(SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as survival_pct
FROM read_parquet('{parquet_path}')
"""
print(con.execute(query_availability).df())

--- 1. Date Span & Row Count ---
  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378

--- 2. The Grain Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain verified: 0 duplicates found. One row exactly equals one content item per client per day.

--- 3. Availability (Survival Check) ---
   total_rows  rows_with_ga4  survival_pct
0     9841378       413966.0           4.2


## 4. Data limits

Unbalanced History: The dataset cannot guarantee uniform historical depth across the inventory.

GSC-Only: The data cannot automatically distinguish between zero traffic and no tracking installed

Window Overlaps: The raw daily table cannot prevent time leakage on it's own.

Causal Proof: This is purely observational data. It can highlight correlations and opportunities, but it cannot prove that a specific content refresh caused a traffic recovery

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.